# 03 - Análise e validação dos clientes
**Squad 2 - Grupo 2**

## Objetivo
Ler a tabela `squad2.ecommerce_clientes` do SQL Server e validar sua estrutura, volume e integridade antes das próximas camadas.

## Validações geradas
Este notebook não gera KPIs. As validações são:
- **Estrutura e amostra:** schema da tabela e 10 linhas de exemplo.
- **Volume:** quantidade de linhas e de colunas.
- **Integridade da chave primária:** quantidade de registros com `id_cliente` nulo, com status "saudável" ou "atenção".

## Saídas
Apenas impressões e tabelas na tela. Nada é gravado.

## Importação das bibliotecas
Leitura do `.env`, caminhos, `os` e as funções do Spark usadas nas validações.

In [0]:
# Importação das bibliotecas necessárias

from dotenv import load_dotenv
from pathlib import Path
import os

from pyspark.sql import functions as F

## Configuração
Carrega o `.env`, lê as credenciais do SQL Server e define a tabela analisada (`squad2.ecommerce_clientes`).

In [0]:
# Localiza e carrega o arquivo .env com as configurações

env_path = next(Path("/Workspace").rglob(".env"), None)

if env_path:
    load_dotenv(env_path)
    print(".env carregado com sucesso.")
else:
    print(".env não encontrado.")

sql_host = os.getenv("SQL_HOST")
sql_database = os.getenv("SQL_DATABASE")
sql_username = os.getenv("SQL_USERNAME")
sql_password = os.getenv("SQL_PASSWORD")

tabela_destino = "squad2.ecommerce_clientes"

print(f"Tabela de análise: {tabela_destino}")

## Leitura da tabela
Lê a tabela de clientes diretamente do SQL Server.

In [0]:
# Lê a tabela de clientes diretamente do SQL Server

df_clientes_analise = spark.read \
    .format("sqlserver") \
    .option("host", sql_host) \
    .option("port", 1433) \
    .option("database", sql_database) \
    .option("dbtable", tabela_destino) \
    .option("user", sql_username) \
    .option("password", sql_password) \
    .load()

print("Tabela lida do SQL Server com sucesso.")

## Estrutura e amostra
Mostra o schema e 10 linhas de exemplo para conferir os tipos e o conteúdo.

In [0]:
# Validação da estrutura e amostragem dos dados

print("=== ESTRUTURA DOS DADOS ===")
df_clientes_analise.printSchema()

print("\n=== AMOSTRA DOS DADOS ===")
display(df_clientes_analise.limit(10))

## Volume de dados
Conta as linhas e as colunas da tabela.

In [0]:
# Validação do volume de dados

quantidade_linhas = df_clientes_analise.count()
quantidade_colunas = len(df_clientes_analise.columns)

print("=== VOLUME DE DADOS ===")
print(f"Linhas   : {quantidade_linhas}")
print(f"Colunas  : {quantidade_colunas}")

## Integridade da chave primária
Conta os registros com `id_cliente` nulo e informa se a chave está saudável ou exige atenção.

In [0]:
# Validação da chave primária id_cliente

nulos_id_cliente = df_clientes_analise.filter(
    F.col("id_cliente").isNull()
).count()

print("=== INTEGRIDADE DA CHAVE PRIMÁRIA ===")
print(f"Registros com id_cliente nulo: {nulos_id_cliente}")

if nulos_id_cliente == 0:
    print("Status: saudável - nenhum registro com id_cliente nulo.")
else:
    print("Status: atenção - existem registros com id_cliente nulo.")